# Stage 2: Rubin blend encoder supervised by the Euclid teacher

Real blends (D2 test split) have no Rubin truth: they are scored against the Euclid teacher. Controlled blends have one: the isolated Rubin stamp of the target and its reference reconstruction.

Paths: **student** (this work), **deblender** (DebVAder paper_baseline stage 2, untouched), **naive** (isolated-galaxy encoder fed the blend), **teacher** (Euclid view through Stage 1). `NB_STAGE2_RUN` picks the run shown in detail.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dp2_cutouts.mlconfig import MLConfig
from dp2_cutouts.ml import figures as F

cfg = MLConfig.load(os.environ['DP2_ML_CONFIG'])
RUN = os.environ.get('NB_STAGE2_RUN', 's2_latent')
IB = list(cfg['rubin']['bands']).index(cfg['rubin']['shape_band'])
runs = [n for n in cfg['stage2']['ablations'] if (cfg.run_dir(n) / 'run.json').exists()]
print('runs found:', runs, '| detail:', RUN)

In [ ]:
F.histories([cfg.run_dir(n) for n in runs], keys=('total_loss', 'latent_loss', 'infonce_loss', 'scene_loss', 'latent_mse'));

## Summary, all runs

In [ ]:
frames = []
for n in runs:
    p = cfg.eval_dir(n) / 'stage2_test_summary.json'
    if not p.exists():
        continue
    s = F.load_json(p)
    for part in ('d2_vs_teacher', 'blends_vs_reference'):
        for path, vals in s.get(part, {}).items():
            frames.append(dict(run=n, set=part, path=path, **vals))
summary = pd.DataFrame(frames)
summary.set_index(['set', 'run', 'path']).sort_index()

## Real blends (D2), against the teacher

In [ ]:
d = pd.read_parquet(cfg.eval_dir(RUN) / 'stage2_d2_test.parquet')
for p in ('student', 'deblender', 'naive'):
    F.scatter_e(d, p, 'teacher', p, 'teacher');

In [ ]:
bins = pd.read_parquet(cfg.eval_dir(RUN) / 'stage2_d2_test_bins.parquet')
F.binned_plot(bins, ['latent_dist_student', 'latent_dist_deblender', 'latent_dist_naive']);

In [ ]:
ex = np.load(cfg.eval_dir(RUN) / 'stage2_d2_test_examples.npz')
F.residual_grid(ex['blend'], [ex['teacher'], ex['student'], ex['deblender']], ['teacher', 'student', 'deblender'], IB, n=8);
F.stamp_grid([ex['euclid']], ['Euclid input of the teacher'], n=8, size=1.2);

## Controlled blends, against truth

`chi2_*_truth`: residual of each reconstruction against the isolated target stamp, per pixel, with its variance. Binned in separation and companion flux ratio.

In [ ]:
p = cfg.eval_dir(RUN) / 'stage2_blends_test.parquet'
if p.exists():
    db = pd.read_parquet(p)
    bb = pd.read_parquet(cfg.eval_dir(RUN) / 'stage2_blends_test_bins.parquet')
    F.binned_plot(bb, ['chi2_student', 'chi2_deblender', 'chi2_naive']);
    F.binned_plot(bb, ['scatter_de1_student', 'scatter_de1_deblender', 'scatter_de1_naive']);
    for q in ('student', 'deblender'):
        F.scatter_e(db, q, 'ref', q, 'reference (target alone)');

In [ ]:
p = cfg.eval_dir(RUN) / 'stage2_blends_test_examples.npz'
if p.exists():
    ex = np.load(p)
    F.stamp_grid([ex['blend'], ex['truth'], ex['ref'], ex['student'], ex['deblender'], ex['naive']],
                 ['blend', 'target alone', 'ref', 'student', 'deblender', 'naive'], n=8);